In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName('etl') \
    .config("spark.jars", "/opt/spark/jars/iceberg-spark-runtime-3.5_2.12-1.6.0.jar") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.spark_catalog.type", "hive") \
    .config("spark.sql.catalog.local.warehouse", "s3a://datalake/iceberg") \
    .getOrCreate()

#Ajuste de log WARN log para ERROR
spark.sparkContext.setLogLevel("ERROR")

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, BooleanType, TimestampType, IntegerType

## Orders Events

In [ ]:
orders_path = '../Data/order_events.jsonl'

order_schema = StructType([
    StructField("event_id", StringType(),  True),
    StructField("user_id", StringType(),  True),
    StructField("order_id", StringType(),  True),  
    StructField("product_id", StringType(),  True),
    StructField("event_type", StringType(),  True),  
    StructField("price", StringType(),  True),
    StructField("region", StringType(),  True),
    StructField("event_timestamp", StringType(),  True),
    StructField("is_delayed", StringType(),  True),
    StructField("is_returned", StringType(),  True),
])


order_df = (
    spark.read    
    .schema(order_schema)             
    .json(orders_path)
)

In [ ]:
order_df.show(5)

In [ ]:
(
    order_df
    .writeTo("iceberg.bronze.tbl_bronze_order_events")
    .createOrReplace()
)

## Products Catalog

In [ ]:
product_catalog_path = '../Data/products.csv'

product_catalog_schema = StructType([
    StructField("product_id", StringType(),  True),
    StructField("product_name", StringType(),  True),
    StructField("category", StringType(),  True),
    StructField("price", StringType(),  True),
])


product_catalog_df = (
    spark.read
    .option("header", True)    
    .option("delimiter", ",")   
    .schema(product_catalog_schema)             
    .csv(product_catalog_path)
)


In [ ]:
product_catalog_df.show(5)

In [ ]:
(
    product_catalog_df
    .writeTo("iceberg.bronze.tbl_bronze_product_catalog")
    .createOrReplace()
)

### Camada Silver

In [ ]:
spark.sql("SHOW TABLES in bronze").toPandas()

In [ ]:
spark.sql("select * from iceberg.bronze.tbl_bronze_product_catalog").show(5)

In [ ]:
spark.sql("""
    CREATE OR REPLACE TABLE iceberg.silver.tbl_silver_product_catalog
    AS
    SELECT
        CAST(product_id AS STRING)          AS product_id,
        CAST(product_name AS STRING)        AS product_name,
        CAST(category AS STRING)            AS category,
        CAST(price AS DOUBLE)               AS price
        
    FROM iceberg.bronze.tbl_bronze_product_catalog

""")

In [ ]:
spark.table("iceberg.silver.tbl_silver_product_catalog").dtypes

In [ ]:
spark.stop()